# World Bank Data360 API — Exploración

---

## Endpoints disponibles (según la spec)

| # | Método | Endpoint | Tag | Descripción |
|---|--------|----------|-----|-------------|
| 1 | `GET`  | `/data360/indicators` | Data | Lista IDs de indicadores de un dataset |
| 2 | `GET`  | `/data360/data` | Data | Obtiene observaciones con filtros |
| 3 | `POST` | `/data360/metadata` | Metadata | Metadata via OData query |
| 4 | `POST` | `/data360/searchv2` | Search | Búsqueda full-text / semántica |
| 5 | `GET`  | `/data360/disaggregation` | Other | Desagregaciones de un indicador |

## 0. Setup

In [22]:
import requests
import pandas as pd
import json
from pprint import pprint

BASE_URL = "https://data360api.worldbank.org"
HEADERS  = {"Content-Type": "application/json", "Accept": "application/json"}

def get(endpoint, params=None):
    url = f"{BASE_URL}{endpoint}"
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    print(f"GET  {url}  →  {r.status_code}")
    r.raise_for_status()
    return r.json()

def post(endpoint, body):
    url = f"{BASE_URL}{endpoint}"
    r = requests.post(url, json=body, headers=HEADERS, timeout=30)
    print(f"POST {url}  →  {r.status_code}")
    r.raise_for_status()
    return r.json()

print("Setup OK ✓")

Setup OK ✓


---
## 1. `GET /data360/indicators` — Qué bases de datos existen

**Parámetro requerido:** `datasetId`  
Devuelve la lista de IDs de indicadores disponibles en un dataset.

In [23]:
# Dataset de ejemplo: World Development Indicators
resp_ind = get("/data360/indicators", params={"datasetId": "WB_WDI"})

print(f"\nTipo de respuesta: {type(resp_ind)}")
if isinstance(resp_ind, dict):
    print(f"Claves: {list(resp_ind.keys())}")
elif isinstance(resp_ind, list):
    print(f"Total indicadores: {len(resp_ind)}")

# Previsualizar
print(json.dumps(resp_ind, indent=2)[:600])

GET  https://data360api.worldbank.org/data360/indicators  →  200

Tipo de respuesta: <class 'list'>
Total indicadores: 1533
[
  "WB_WDI_AG_CON_FERT_ZS",
  "WB_WDI_AG_LND_EL5M_UR_K2",
  "WB_WDI_BM_GSR_GNFS_CD",
  "WB_WDI_BM_GSR_INSF_ZS",
  "WB_WDI_BM_GSR_NFSV_CD",
  "WB_WDI_BN_CAB_XOKA_CD",
  "WB_WDI_BX_GSR_FCTY_CD",
  "WB_WDI_DC_DAC_BELL_CD",
  "WB_WDI_DC_DAC_GRCL_CD",
  "WB_WDI_DC_DAC_NORL_CD",
  "WB_WDI_DC_DAC_TOTL_CD",
  "WB_WDI_DC_ODA_TLDC_GN_ZS",
  "WB_WDI_DT_NFL_OFFT_CD",
  "WB_WDI_DT_NFL_PCBO_CD",
  "WB_WDI_DT_NFL_SDGF_CD",
  "WB_WDI_DT_NFL_UNPB_CD",
  "WB_WDI_DT_ODA_ALLD_CD",
  "WB_WDI_DT_ODA_ODAT_KD",
  "WB_WDI_DT_TDS_DECT_GN_ZS",
  "WB_WDI_DT_TDS_DIMF_CD",
  "WB_WDI_DT_TDS_DPPG_GN_ZS",
  "WB_WDI_EG_CFT_AC


In [24]:
# Normalizar a lista
if isinstance(resp_ind, list):
    ind_list = resp_ind
elif isinstance(resp_ind, dict):
    ind_list = resp_ind.get("value", resp_ind.get("data", resp_ind.get("results", [])))

print(f"Total indicadores en WB_WDI: {len(ind_list)}")
print("\nPrimeros 10:")
for x in ind_list[:10]:
    print(" ", x)

Total indicadores en WB_WDI: 1533

Primeros 10:
  WB_WDI_AG_CON_FERT_ZS
  WB_WDI_AG_LND_EL5M_UR_K2
  WB_WDI_BM_GSR_GNFS_CD
  WB_WDI_BM_GSR_INSF_ZS
  WB_WDI_BM_GSR_NFSV_CD
  WB_WDI_BN_CAB_XOKA_CD
  WB_WDI_BX_GSR_FCTY_CD
  WB_WDI_DC_DAC_BELL_CD
  WB_WDI_DC_DAC_GRCL_CD
  WB_WDI_DC_DAC_NORL_CD


In [25]:
# Convertir a DataFrame
if isinstance(ind_list[0], str):
    df_ind = pd.DataFrame(ind_list, columns=["INDICATOR_ID"])
else:
    df_ind = pd.json_normalize(ind_list)

print(f"Shape: {df_ind.shape}")
display(df_ind.head(20))

Shape: (1533, 1)


,INDICATOR_ID
0,WB_WDI_AG_CON_FERT_ZS
1,WB_WDI_AG_LND_EL5M_UR_K2
2,WB_WDI_BM_GSR_GNFS_CD
3,WB_WDI_BM_GSR_INSF_ZS
4,WB_WDI_BM_GSR_NFSV_CD
5,WB_WDI_BN_CAB_XOKA_CD
6,WB_WDI_BX_GSR_FCTY_CD
7,WB_WDI_DC_DAC_BELL_CD
8,WB_WDI_DC_DAC_GRCL_CD
9,WB_WDI_DC_DAC_NORL_CD


In [26]:
# Guardar un indicador de ejemplo para las siguientes secciones
if isinstance(ind_list[0], str):
    IND_EJEMPLO = ind_list[0]
else:
    id_col = next((k for k in ind_list[0] if 'id' in k.lower() or 'code' in k.lower()), list(ind_list[0].keys())[0])
    IND_EJEMPLO = ind_list[0][id_col]

print(f"Indicador para siguientes pasos: {IND_EJEMPLO}")

Indicador para siguientes pasos: WB_WDI_AG_CON_FERT_ZS


---
## 2. `POST /data360/metadata` — Qué indicadores existen y cómo se describen

Body: objeto JSON con clave `query` (sintaxis OData: `$filter`, `$select`).

**Ejemplos oficiales de la spec:**
```json
// Todos los campos de un indicador
{"query": "&$filter=series_description/idno eq 'WB_WDI_SP_POP_TOTL'"}

// Campos selectivos
{"query": "&$filter=series_description/idno eq 'WB_WDI_SP_POP_TOTL'&$select=series_description/database_id,series_description/idno"}

// Todos los indicadores de un database
{"query": "&$filter=series_description/database_id eq 'WB_WDI'&$select=series_description/database_id,series_description/idno"}
```

In [27]:
# --- Ejemplo 1 (oficial): todos los campos de un indicador ---
meta1 = post("/data360/metadata", body={
    "query": "&$filter=series_description/idno eq 'WB_WDI_SP_POP_TOTL'"
})

print(f"Claves: {list(meta1.keys()) if isinstance(meta1, dict) else type(meta1)}")
print(json.dumps(meta1, indent=2)[:2000])

POST https://data360api.worldbank.org/data360/metadata  →  200
Claves: ['@odata.context', '@odata.count', 'value']
{
  "@odata.context": "https://itsda-dataexp-prd.search.windows.net/indexes('data360-metadata')/$metadata#docs(*)",
  "@odata.count": 1,
  "value": [
    {
      "@search.score": 1.0,
      "id": "META_WB_WDI_SP_POP_TOTL",
      "idno": "141c1764-39a9-44d4-9c5d-378ae3a4fbb6",
      "type": "indicator",
      "subtype": "timeseries",
      "disaggregation_types": [],
      "isDelete": null,
      "cfPath": null,
      "doc_type": null,
      "remove_chart_type": "stackedBar, pie",
      "data_confidentiality_code": "PU",
      "data_confidentiality_name": "Public",
      "dsd_name": "DATA360",
      "dsd_version": "1.3",
      "dsd_codelist": [
        "DATA_SOURCE",
        "FREQ",
        "REF_AREA",
        "INDICATOR",
        "SEX",
        "AGE",
        "URBANISATION",
        "UNIT_MEASURE",
        "COMP_BREAKDOWN_1",
        "COMP_BREAKDOWN_2",
        "COMP_BREAK

In [ ]:
# --- Ejemplo 2 campos específicos de un indicador ---
meta2 = post("/data360/metadata", body={
    "query": "&$filter=series_description/idno eq 'WB_WDI_SP_POP_TOTL'"
            "&$select=series_description/database_id,series_description/idno"
})
print(json.dumps(meta2, indent=2))

POST https://data360api.worldbank.org/data360/metadata  →  200
{
  "@odata.context": "https://itsda-dataexp-prd.search.windows.net/indexes('data360-metadata')/$metadata#docs(*)",
  "@odata.count": 1,
  "value": [
    {
      "@search.score": 1.0,
      "series_description": {
        "idno": "WB_WDI_SP_POP_TOTL",
        "database_id": "WB_WDI"
      }
    }
  ]
}


In [ ]:
# --- Ejemplo 3 todos los indicadores de WB_WDI ---
meta3 = post("/data360/metadata", body={
    "query": "&$filter=series_description/database_id eq 'WB_WDI'"
            "&$select=series_description/database_id,series_description/idno"
})

if isinstance(meta3, dict):
    if "count" in meta3:
        print(f"Total: {meta3['count']}")
    items = meta3.get("value", meta3.get("data", []))
    print(f"Registros en esta página: {len(items)}")
    if items:
        df_meta3 = pd.json_normalize(items)
        display(df_meta3.head(20))

POST https://data360api.worldbank.org/data360/metadata  →  200
Registros en esta página: 50


,@search.score,series_description.idno,series_description.database_id
0,1.0,WB_WDI_PER_SA_ALLSA_ADQ_POP_TOT,WB_WDI
1,1.0,WB_WDI_BX_GSR_NFSV_CD,WB_WDI
2,1.0,WB_WDI_BX_GSR_TOTL_CD,WB_WDI
3,1.0,WB_WDI_BX_KLT_DINV_CD_WD,WB_WDI
4,1.0,WB_WDI_BX_KLT_DINV_WD_GD_ZS,WB_WDI
5,1.0,WB_WDI_CC_PER_RNK,WB_WDI
6,1.0,WB_WDI_CC_STD_ERR,WB_WDI
7,1.0,WB_WDI_AG_PRD_FOOD_XD,WB_WDI
8,1.0,WB_WDI_AG_YLD_CREL_KG,WB_WDI
9,1.0,WB_WDI_BM_GSR_ROYL_CD,WB_WDI


---
## 3. `POST /data360/searchv2` — Buscar indicadores por tema

Schema `SearchQuerySchema`:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `search` | string | Término de búsqueda |
| `select` | string | Campos a retornar (CSV) |
| `filter` | string | Filtro OData |
| `orderby` | string | Orden de resultados |
| `top` | int | Máx resultados |
| `skip` | int | Paginación |
| `count` | bool | Incluir total en respuesta |
| `facets` | array | Facetas para agrupar resultados |

In [30]:
# --- Búsqueda simple (ejemplo oficial de la spec) ---
search1 = post("/data360/searchv2", body={
    "count": True,
    "select": "series_description/idno, series_description/name, series_description/database_id",
    "search": "poverty",
    "top": 10
})

print(f"Claves de respuesta: {list(search1.keys()) if isinstance(search1, dict) else type(search1)}")

# Total de resultados
total = search1.get("@odata.count", search1.get("count", "?"))
print(f"Total de indicadores sobre 'poverty': {total}")

POST https://data360api.worldbank.org/data360/searchv2  →  200
Claves de respuesta: ['@odata.context', '@odata.count', 'value']
Total de indicadores sobre 'poverty': 1498


In [31]:
items1 = search1.get("value", search1.get("data", []))
print(f"Resultados recibidos: {len(items1)}")

if items1:
    df_s1 = pd.json_normalize(items1)
    print(f"Columnas: {list(df_s1.columns)}")
    display(df_s1)

Resultados recibidos: 10
Columnas: ['@search.score', 'series_description.idno', 'series_description.name', 'series_description.database_id']


,@search.score,series_description.idno,series_description.name,series_description.database_id
0,43.175850,WB_SSGD_POVERTY_RATIO_NPL,Poverty headcount ratio at national poverty lines,WB_SSGD
1,43.002720,WB_WDI_SI_POV_GAPS,Poverty gap at $3.00 a day (2021 PPP) (%),WB_WDI
2,42.695595,WB_WDI_SI_POV_NAHC,Poverty headcount ratio at national poverty li...,WB_WDI
3,42.278190,WB_WDI_SI_POV_LMIC,Poverty headcount ratio at $4.20 a day (2021 P...,WB_WDI
4,40.758580,WB_PIP_HEADCOUNT_IPL,"Poverty headcount ratio (2021 PPP, $3.00) (% o...",WB_PIP
5,40.028164,WB_WDI_SI_POV_UMIC,Poverty headcount ratio at $8.30 a day (2021 P...,WB_WDI
6,38.672024,WB_WDI_SI_POV_SOPO,Poverty headcount ratio at societal poverty li...,WB_WDI
7,37.627450,WB_WDI_SI_POV_MPWB,Multidimensional poverty headcount ratio (Worl...,WB_WDI
8,37.424843,WB_WDI_SI_POV_LMIC_GP,Poverty gap at $4.20 a day (2021 PPP) (%),WB_WDI
9,37.358562,WB_PIP_NPOOR_IPL,Population living below the poverty line (2021...,WB_PIP


In [32]:
# --- Búsqueda avanzada: filtrar por topic (ejemplo oficial de la spec) ---
search2 = post("/data360/searchv2", body={
    "count": False,
    "filter": "series_description/topics/any(t: t/name eq 'Health') and type eq 'indicator'",
    "orderby": "series_description/name",
    "select": "series_description/idno, series_description/name, series_description/database_id",
    "search": "nutrition",
    "top": 20,
    "skip": 0
})

items2 = search2.get("value", search2.get("data", []))
print(f"Resultados Health + nutrition: {len(items2)}")
if items2:
    display(pd.json_normalize(items2))

POST https://data360api.worldbank.org/data360/searchv2  →  200
Resultados Health + nutrition: 13


,@search.score,series_description.idno,series_description.name,series_description.database_id
0,6.935638,WB_GS_SH_DTH_ZS,Cause of death (%),WB_GS
1,1.864711,WB_HCP_VACDTP3,"DTP vaccination rate, third dose (%)",WB_HCP
2,2.548645,HD_HCIP_STNT,Fraction of Children Under 5 Not Stunted,WB_HCIP
3,3.720422,WB_HCP_NOSTU,Fraction of Children Under 5 Not Stunted (%),WB_HCP
4,2.194086,HD_HCIP_OVRL,Human Capital Index Plus (HCI+),WB_HCIP
5,13.465480,HD_HCIP_HLTH,Human Capital Index Plus (HCI+): Health Pillar...,WB_HCIP
6,9.383695,WB_HCP_MEALFREQ,"Minimum meal frequency (%), ages 6-23 months",WB_HCP
7,2.509007,WB_HCP_FAO_WASTING,Percentage of children under 5 years affected ...,WB_HCP
8,3.298883,WB_SSGD_PCT_POP_HEALTH_INSURANCE,Percentage of population with health insurance,WB_SSGD
9,2.885274,WB_HCP_HYPERTENSION,"Prevalence of hypertension (%), ages 30-79",WB_HCP


In [33]:
# --- Usar facets para descubrir qué databases y tipos existen ---
search_facets = post("/data360/searchv2", body={
    "count": True,
    "select": "series_description/idno, series_description/database_id",
    "search": "*",
    "top": 1,
    "facets": ["series_description/database_id", "type"]
})

print("Respuesta con facets:")
print(json.dumps(search_facets, indent=2)[:3000])

POST https://data360api.worldbank.org/data360/searchv2  →  200
Respuesta con facets:
{
  "@odata.context": "https://itsda-dataexp-prd.search.windows.net/indexes('data360-metadata-vector')/$metadata#docs(*)",
  "@odata.count": 12947,
  "@search.facets": {
    "type": [
      {
        "value": "indicator",
        "count": 10228
      },
      {
        "value": "analytics",
        "count": 2151
      },
      {
        "value": "document",
        "count": 188
      },
      {
        "value": "dataset",
        "count": 161
      },
      {
        "value": "insights-resources",
        "count": 89
      },
      {
        "value": "country factsheets",
        "count": 85
      },
      {
        "value": "global report",
        "count": 45
      }
    ],
    "series_description/database_id": [
      {
        "value": "WB_WDI",
        "count": 1534
      },
      {
        "value": "IMF_BOP",
        "count": 1147
      },
      {
        "value": "WB_EDSTATS",
        "count": 1

---
## 4. `GET /data360/data` — Descargar observaciones

**Parámetros según la spec:**

| Parámetro | Requerido | Descripción |
|-----------|:---------:|-------------|
| `DATABASE_ID` | ✅ | ID del dataset, ej: `WB_WDI` |
| `INDICATOR` | — | Código del indicador |
| `REF_AREA` | — | Código ISO3 del país/región |
| `TIME_PERIOD` | — | Período exacto |
| `timePeriodFrom` | — | Inicio del rango temporal |
| `timePeriodTo` | — | Fin del rango temporal |
| `FREQ` | — | Frecuencia: `A`=anual, `Q`=trimestral, `M`=mensual |
| `SEX` | — | `M` / `F` / `_T` (total) |
| `AGE` | — | Grupo etario o `_T` (total) |
| `URBANISATION` | — | `U`=urbano, `R`=rural, `_T`=total |
| `COMP_BREAKDOWN_1/2/3` | — | Dimensiones específicas del indicador |
| `UNIT_MEASURE` | — | Unidad de medida |
| `UNIT_MULT` | — | Multiplicador |
| `skip` | — | Paginación (máx 1000 registros/llamada) |

**Estructura de respuesta** (`DataResponse`):
```json
{
  "count": 7204,
  "value": [
    {
      "OBS_VALUE": "38.7168",   "DATABASE_ID": "WB_WDI",
      "INDICATOR": "...",       "REF_AREA": "ARB",
      "TIME_PERIOD": "1970",    "FREQ": "A",
      "SEX": "F",               "AGE": "_T",
      "URBANISATION": "_T",     "OBS_STATUS": "A",
      "OBS_CONF": "PU",         "LATEST_DATA": false, ...
    }
  ]
}
```

In [34]:
# --- Población total de Argentina, 2000-2023 ---
data1 = get("/data360/data", params={
    "DATABASE_ID":    "WB_WDI",
    "INDICATOR":      "WB_WDI_SP_POP_TOTL",
    "REF_AREA":       "ARG",
    "timePeriodFrom": "2000",
    "timePeriodTo":   "2023",
    "FREQ":           "A",
    "skip":           0
})

print(f"Total observaciones: {data1.get('count', '?')}")
obs1 = data1.get("value", [])
print(f"En esta página: {len(obs1)}")

GET  https://data360api.worldbank.org/data360/data  →  200
Total observaciones: 24
En esta página: 24


In [35]:
# Estructura de una observación
if obs1:
    print("Una observación tiene las siguientes claves:")
    pprint(obs1[0])

Una observación tiene las siguientes claves:
{'AGE': '_T',
 'AGG_METHOD': '_Z',
 'COMMENT_OBS': None,
 'COMMENT_TS': None,
 'COMP_BREAKDOWN_1': '_Z',
 'COMP_BREAKDOWN_2': '_Z',
 'COMP_BREAKDOWN_3': '_Z',
 'DATABASE_ID': 'WB_WDI',
 'DATA_SOURCE': None,
 'DECIMALS': '2',
 'FREQ': 'A',
 'INDICATOR': 'WB_WDI_SP_POP_TOTL',
 'LATEST_DATA': False,
 'OBS_CONF': 'PU',
 'OBS_STATUS': 'A',
 'OBS_VALUE': '40854831',
 'REF_AREA': 'ARG',
 'SEX': '_T',
 'TIME_FORMAT': 'P1Y',
 'TIME_PERIOD': '2009',
 'UNIT_MEASURE': 'PS',
 'UNIT_MULT': 0,
 'UNIT_TYPE': None,
 'URBANISATION': '_T'}


In [36]:
# Convertir a DataFrame y explorar
df1 = pd.DataFrame(obs1)
df1["OBS_VALUE"]   = pd.to_numeric(df1["OBS_VALUE"], errors="coerce")
df1["TIME_PERIOD"] = pd.to_numeric(df1["TIME_PERIOD"], errors="coerce")

print(f"Columnas: {list(df1.columns)}")
display(df1[["TIME_PERIOD","REF_AREA","INDICATOR","OBS_VALUE",
             "FREQ","SEX","AGE","URBANISATION","UNIT_MEASURE",
             "OBS_STATUS","LATEST_DATA"]].sort_values("TIME_PERIOD"))

Columnas: ['OBS_VALUE', 'TIME_FORMAT', 'UNIT_MULT', 'COMMENT_OBS', 'OBS_STATUS', 'OBS_CONF', 'AGG_METHOD', 'DECIMALS', 'COMMENT_TS', 'DATA_SOURCE', 'LATEST_DATA', 'DATABASE_ID', 'INDICATOR', 'REF_AREA', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_3', 'TIME_PERIOD', 'FREQ', 'UNIT_MEASURE', 'UNIT_TYPE']


,TIME_PERIOD,REF_AREA,INDICATOR,OBS_VALUE,FREQ,SEX,AGE,URBANISATION,UNIT_MEASURE,OBS_STATUS,LATEST_DATA
22,2000,ARG,WB_WDI_SP_POP_TOTL,37213984,A,_T,_T,_T,PS,A,False
4,2001,ARG,WB_WDI_SP_POP_TOTL,37624825,A,_T,_T,_T,PS,A,False
13,2002,ARG,WB_WDI_SP_POP_TOTL,38029349,A,_T,_T,_T,PS,A,False
2,2003,ARG,WB_WDI_SP_POP_TOTL,38424282,A,_T,_T,_T,PS,A,False
14,2004,ARG,WB_WDI_SP_POP_TOTL,38815916,A,_T,_T,_T,PS,A,False
3,2005,ARG,WB_WDI_SP_POP_TOTL,39216789,A,_T,_T,_T,PS,A,False
1,2006,ARG,WB_WDI_SP_POP_TOTL,39622115,A,_T,_T,_T,PS,A,False
10,2007,ARG,WB_WDI_SP_POP_TOTL,40016763,A,_T,_T,_T,PS,A,False
11,2008,ARG,WB_WDI_SP_POP_TOTL,40424148,A,_T,_T,_T,PS,A,False
0,2009,ARG,WB_WDI_SP_POP_TOTL,40854831,A,_T,_T,_T,PS,A,False


In [37]:
# --- Comparar múltiples países con paginación usando skip ---
# La spec indica máximo 1000 registros por llamada; se usa skip para paginar

paises = ["ARG", "BRA", "CHL", "MEX", "COL", "PER"]
all_obs = []

for pais in paises:
    r = get("/data360/data", params={
        "DATABASE_ID":    "WB_WDI",
        "INDICATOR":      "WB_WDI_SP_POP_TOTL",
        "REF_AREA":       pais,
        "timePeriodFrom": "2010",
        "timePeriodTo":   "2023",
        "FREQ":           "A",
        "skip":           0
    })
    all_obs.extend(r.get("value", []))

df_latam = pd.DataFrame(all_obs)
df_latam["OBS_VALUE"]   = pd.to_numeric(df_latam["OBS_VALUE"], errors="coerce")
df_latam["TIME_PERIOD"] = pd.to_numeric(df_latam["TIME_PERIOD"], errors="coerce")

pivot = df_latam.pivot_table(index="TIME_PERIOD", columns="REF_AREA", values="OBS_VALUE")
print("Población total LATAM (habitantes):")
display(pivot)

GET  https://data360api.worldbank.org/data360/data  →  200
GET  https://data360api.worldbank.org/data360/data  →  200
GET  https://data360api.worldbank.org/data360/data  →  200
GET  https://data360api.worldbank.org/data360/data  →  200
GET  https://data360api.worldbank.org/data360/data  →  200
GET  https://data360api.worldbank.org/data360/data  →  200
Población total LATAM (habitantes):


REF_AREA,ARG,BRA,CHL,COL,MEX,PER
TIME_PERIOD,,,,,,
2010,41288694.0,193701929.0,17181464.0,44777319.0,113623895.0,29086019.0
2011,41730660.0,195284734.0,17351816.0,45259614.0,115243504.0,29304086.0
2012,42161721.0,196876111.0,17519119.0,45715810.0,116818208.0,29550366.0
2013,42582455.0,198478299.0,17687006.0,46151584.0,118343573.0,29817919.0
2014,43024071.0,200085127.0,17864195.0,46565429.0,119784261.0,30115826.0
2015,43477012.0,201675532.0,18047625.0,46969940.0,121072306.0,30457600.0
2016,43900313.0,203218114.0,18267221.0,47437512.0,122251351.0,30866494.0
2017,44288894.0,204703445.0,18558868.0,48131078.0,123400057.0,31324637.0
2018,44654882.0,206107261.0,18893191.0,49024465.0,124573711.0,31897584.0


---
## 5. `GET /data360/disaggregation` — Estructura dimensional de un indicador

**Parámetros requeridos:** `datasetId` + `indicatorId`  
Devuelve todos los valores posibles de SEX, AGE, URBANISATION, COMP_BREAKDOWN_1/2/3 para ese indicador.

In [38]:
# --- Desagregaciones del indicador Población Total ---
disag1 = get("/data360/disaggregation", params={
    "datasetId":   "WB_WDI",
    "indicatorId": "WB_WDI_SP_POP_TOTL"
})

print(f"Tipo: {type(disag1)}")
print(json.dumps(disag1, indent=2)[:3000])

GET  https://data360api.worldbank.org/data360/disaggregation  →  200
Tipo: <class 'list'>
[
  {
    "field_name": "FREQ",
    "label_name": "FREQ",
    "field_value": [
      "A"
    ]
  },
  {
    "field_name": "REF_AREA",
    "label_name": "REF_AREA",
    "field_value": [
      "ABW",
      "AFE",
      "AFG",
      "AFW",
      "AGO",
      "ALB",
      "AND",
      "ARB",
      "ARE",
      "ARG",
      "ARM",
      "ASM",
      "ATG",
      "AUS",
      "AUT",
      "AZE",
      "BDI",
      "BEL",
      "BEN",
      "BFA",
      "BGD",
      "BGR",
      "BHR",
      "BHS",
      "BIH",
      "BLR",
      "BLZ",
      "BMU",
      "BOL",
      "BRA",
      "BRB",
      "BRN",
      "BTN",
      "BWA",
      "CAF",
      "CAN",
      "CEB",
      "CHE",
      "CHI",
      "CHL",
      "CHN",
      "CIV",
      "CMR",
      "COD",
      "COG",
      "COL",
      "COM",
      "CPV",
      "CRI",
      "CSS",
      "CUB",
      "CUW",
      "CYM",
      "CYP",
      "CZE",
      "DEU

In [39]:
# Convertir a DataFrame
if isinstance(disag1, list):
    df_disag1 = pd.json_normalize(disag1)
elif isinstance(disag1, dict):
    items = disag1.get("value", disag1.get("data", [disag1]))
    df_disag1 = pd.json_normalize(items)

print(f"Columnas: {list(df_disag1.columns)}")
display(df_disag1)

Columnas: ['field_name', 'label_name', 'field_value']


,field_name,label_name,field_value
0,FREQ,FREQ,[A]
1,REF_AREA,REF_AREA,"[ABW, AFE, AFG, AFW, AGO, ALB, AND, ARB, ARE, ..."
2,INDICATOR,INDICATOR,[WB_WDI_SP_POP_TOTL]
3,SEX,SEX,[_T]
4,AGE,AGE,[_T]
5,URBANISATION,URBANISATION,[_T]
6,UNIT_MEASURE,UNIT_MEASURE,[PS]
7,COMP_BREAKDOWN_1,COMP_BREAKDOWN_1,[_Z]
8,COMP_BREAKDOWN_2,COMP_BREAKDOWN_2,[_Z]
9,COMP_BREAKDOWN_3,COMP_BREAKDOWN_3,[_Z]


In [40]:
# --- Indicador con desagregación por sexo: tasa de completitud escolar ---
disag2 = get("/data360/disaggregation", params={
    "datasetId":   "WB_WDI",
    "indicatorId": "WB_WDI_SE_PRM_CMPT_FE_ZS"
})

print("Desagregaciones de WB_WDI_SE_PRM_CMPT_FE_ZS:")
print(json.dumps(disag2, indent=2)[:2000])

GET  https://data360api.worldbank.org/data360/disaggregation  →  200
Desagregaciones de WB_WDI_SE_PRM_CMPT_FE_ZS:
[
  {
    "field_name": "FREQ",
    "label_name": "Frequency of observation",
    "field_value": [
      "A"
    ]
  },
  {
    "field_name": "REF_AREA",
    "label_name": "Reference area",
    "field_value": [
      "ABW",
      "AFE",
      "AFG",
      "AFW",
      "AGO",
      "ALB",
      "AND",
      "ARB",
      "ARE",
      "ARG",
      "ARM",
      "ATG",
      "AUT",
      "AZE",
      "BDI",
      "BEN",
      "BFA",
      "BGD",
      "BGR",
      "BHR",
      "BHS",
      "BIH",
      "BLR",
      "BLZ",
      "BMU",
      "BOL",
      "BRB",
      "BRN",
      "BTN",
      "BWA",
      "CAF",
      "CAN",
      "CEB",
      "CHE",
      "CHL",
      "CHN",
      "CIV",
      "CMR",
      "COD",
      "COG",
      "COL",
      "COM",
      "CPV",
      "CRI",
      "CSS",
      "CUB",
      "CYM",
      "CYP",
      "CZE",
      "DEU",
      "DMA",
      "DNK",

---
## 6. Resumen: Mapa completo de la API

In [41]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║           WORLD BANK DATA360 API — Estructura de datos              ║
╠══════════════════════════════════════════════════════════════════════╣
║  Base URL: https://data360api.worldbank.org                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  ENDPOINTS                                                           ║
║  ──────────────────────────────────────────────────────────────────  ║
║  GET  /data360/indicators      Lista de IDs de un dataset            ║
║       Req: datasetId (ej: WB_WDI)                                   ║
║                                                                      ║
║  POST /data360/metadata        Metadata via OData                    ║
║       Body: {"query": "&$filter=...&$select=..."}                   ║
║                                                                      ║
║  POST /data360/searchv2        Búsqueda full-text / semántica        ║
║       Body: {search, select, filter, top, skip, count, facets}      ║
║                                                                      ║
║  GET  /data360/data            Observaciones filtradas               ║
║       Req: DATABASE_ID                                               ║
║       Opt: INDICATOR, REF_AREA, TIME_PERIOD, timePeriodFrom/To,     ║
║            FREQ, SEX, AGE, URBANISATION, COMP_BREAKDOWN_1/2/3,      ║
║            UNIT_MEASURE, UNIT_MULT, skip                            ║
║                                                                      ║
║  GET  /data360/disaggregation  Dimensiones de un indicador          ║
║       Req: datasetId + indicatorId                                  ║
╠══════════════════════════════════════════════════════════════════════╣
║  ESTRUCTURA DE UNA OBSERVACIÓN (DataResponse.value[])               ║
║  ──────────────────────────────────────────────────────────────────  ║
║  DATABASE_ID       ID del dataset          WB_WDI                   ║
║  INDICATOR         Código del indicador    WB_WDI_SP_POP_TOTL       ║
║  REF_AREA          País / región (ISO3)    ARG                       ║
║  TIME_PERIOD       Año / período           2023                      ║
║  FREQ              Frecuencia              A / Q / M                 ║
║  SEX               Sexo                    M / F / _T                ║
║  AGE               Grupo etario            _T (total) u otro        ║
║  URBANISATION      Urbanización            U / R / _T                ║
║  COMP_BREAKDOWN_1  Dimensión específica    _Z (no aplica) u otro    ║
║  OBS_VALUE         El dato                 "45123456"                ║
║  OBS_STATUS        Estado del dato         A / E / P                 ║
║  OBS_CONF          Confidencialidad        PU (público)             ║
║  LATEST_DATA       ¿Es el más reciente?    true / false             ║
║  UNIT_MEASURE      Unidad                  PT=% IX=índice etc.      ║
║  UNIT_MULT         Multiplicador           0=unidades 3=miles       ║
║  DATA_SOURCE       Fuente original         WB_WDI                   ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║           WORLD BANK DATA360 API — Estructura de datos              ║
╠══════════════════════════════════════════════════════════════════════╣
║  Base URL: https://data360api.worldbank.org                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  ENDPOINTS                                                           ║
║  ──────────────────────────────────────────────────────────────────  ║
║  GET  /data360/indicators      Lista de IDs de un dataset            ║
║       Req: datasetId (ej: WB_WDI)                                   ║
║                                                                      ║
║  POST /data360/metadata        Metadata via OData                    ║
║       Body: {"query": "&$filter=...&$select=..."}                   ║
║                                                                      ║
║  POST /data360/searchv2        Búsqueda full-text / 

In [42]:
# --- Función de utilidad: descarga paginada completa ---

def fetch_all(database_id, indicator, ref_area=None,
              period_from=None, period_to=None, freq="A"):
    """
    Descarga todas las páginas del endpoint GET /data360/data.
    La spec indica máximo 1000 registros por llamada; usa `skip` para paginar.
    """
    all_records = []
    skip = 0

    while True:
        params = {
            "DATABASE_ID": database_id,
            "INDICATOR":   indicator,
            "FREQ":        freq,
            "skip":        skip,
        }
        if ref_area:    params["REF_AREA"]       = ref_area
        if period_from: params["timePeriodFrom"] = period_from
        if period_to:   params["timePeriodTo"]   = period_to

        resp    = get("/data360/data", params=params)
        records = resp.get("value", [])
        total   = resp.get("count", 0)

        all_records.extend(records)
        print(f"  skip={skip}: +{len(records)} registros  (total {len(all_records)}/{total})")

        if len(records) < 1000 or len(all_records) >= total:
            break
        skip += 1000

    df = pd.DataFrame(all_records)
    if not df.empty:
        df["OBS_VALUE"]   = pd.to_numeric(df["OBS_VALUE"], errors="coerce")
        df["TIME_PERIOD"] = pd.to_numeric(df["TIME_PERIOD"], errors="coerce")
    return df


# Ejemplo: PIB per cápita Argentina
df_pib = fetch_all(
    database_id  = "WB_WDI",
    indicator    = "WB_WDI_NY_GDP_PCAP_CD",
    ref_area     = "ARG",
    period_from  = "2000",
    period_to    = "2023"
)

print(f"\nShape: {df_pib.shape}")
display(df_pib[["TIME_PERIOD","REF_AREA","INDICATOR","OBS_VALUE","UNIT_MEASURE"]].sort_values("TIME_PERIOD"))

GET  https://data360api.worldbank.org/data360/data  →  200
  skip=0: +24 registros  (total 24/24)

Shape: (24, 24)


,TIME_PERIOD,REF_AREA,INDICATOR,OBS_VALUE,UNIT_MEASURE
22,2000,ARG,WB_WDI_NY_GDP_PCAP_CD,7637.014892,USD
3,2001,ARG,WB_WDI_NY_GDP_PCAP_CD,7141.475077,USD
13,2002,ARG,WB_WDI_NY_GDP_PCAP_CD,2569.699635,USD
2,2003,ARG,WB_WDI_NY_GDP_PCAP_CD,3320.477751,USD
14,2004,ARG,WB_WDI_NY_GDP_PCAP_CD,4242.020991,USD
10,2005,ARG,WB_WDI_NY_GDP_PCAP_CD,5067.653423,USD
1,2006,ARG,WB_WDI_NY_GDP_PCAP_CD,5869.380290,USD
9,2007,ARG,WB_WDI_NY_GDP_PCAP_CD,7185.251551,USD
11,2008,ARG,WB_WDI_NY_GDP_PCAP_CD,8944.110266,USD
0,2009,ARG,WB_WDI_NY_GDP_PCAP_CD,8150.235270,USD
